# 7주차 워크북 2 — PDF RAG 질의응답과 Gradio 웹 화면

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itlects/2026-1_ai-convergence-apply/blob/main/Week07/Week07_Workbook2_Student.ipynb)

**학습목표**
1. 워크북 1의 검색기로 근거 청크를 찾고, 임계값으로 근거 없는 질문을 걸러 낸다.
2. 검색 근거와 지시문으로 프롬프트를 구성한다 (RAG의 G 단계).
3. (선택) 수업용 OpenAI API 키로 근거 기반 답변을 생성한다.
4. Gradio로 질문 입력 → 답변·근거 표시 웹 화면을 만든다.

**API 키가 없어도** 모든 셀이 실행됩니다. 이 경우 '검색 모드'로 근거만 보여 줍니다.

## 미리 보는 핵심 키워드

| 키워드 | 역할 |
| --- | --- |
| `THRESHOLD` | 최고 점수가 이 값보다 낮으면 "문서에서 확인할 수 없습니다" (워크북 1에서 0.16 도출) |
| `build_prompt(question, hits)` | 근거 청크에 번호·페이지를 붙여 LLM 입력 문장을 만듦 |
| `userdata.get("OPENAI_API_KEY")` | Colab Secrets(왼쪽 열쇠 아이콘)에 저장한 키 읽기 |
| `getpass()` | 입력 내용이 화면·노트북에 남지 않는 키 입력창 |
| `OpenAI(api_key=...)` | OpenAI API 클라이언트 생성 |
| `client.responses.create(model, instructions, input)` | 지시문(instructions)과 질문·근거(input)로 답변 요청 |
| `response.output_text` | 생성된 답변 문자열 |
| `temperature` | 낮을수록 같은 질문에 비슷한 답 (0.0~2.0) |
| `gr.Interface(fn, inputs, outputs)` | Python 함수를 웹 입력·출력 화면과 연결 |
| `gr.Textbox` / `gr.Checkbox` / `gr.Slider` / `gr.Markdown` | 문장 입력, 참/거짓 선택, 숫자 선택, 서식 있는 출력 |
| `app.launch()` | 웹 화면 실행 (Colab에서는 셀 아래에 표시) |

## 0. 데이터 준비 (Google Drive 자동 다운로드)

In [ ]:
# 0. 실습 데이터 준비
# 강의자가 Google Drive에 미리 올려 둔 PDF·CSV를 파일 ID로 내려받습니다.
# 순서: ① Google Drive → ② GitHub 백업 → ③ 직접 업로드
import urllib.request
from pathlib import Path

DATA_DIR = Path("week07_data")
DATA_DIR.mkdir(exist_ok=True)

PDF_PATH = DATA_DIR / "week07_course_guide.pdf"
CSV_PATH = DATA_DIR / "week07_faq.csv"

# Drive 파일 ID (공유: 링크가 있는 모든 사용자 - 보기)
PDF_DOC_ID = "1WKmS8gV7-a4S1RjVifNo-dsLhN3bIdhyv4asO3B3qX8"
CSV_FILE_ID = "1K_rzSWJhPd7skEyyh-3rK8SRMSknwMpS"

GITHUB_RAW = (
    "https://raw.githubusercontent.com/itlects/"
    "2026-1_ai-convergence-apply/main/Week07/data/"
)

SOURCES = {
    PDF_PATH: [
        "https://docs.google.com/document/d/"
        + PDF_DOC_ID + "/export?format=pdf",
        GITHUB_RAW + PDF_PATH.name,
    ],
    CSV_PATH: [
        "https://drive.google.com/uc?export=download&id="
        + CSV_FILE_ID,
        GITHUB_RAW + CSV_PATH.name,
    ],
}


def looks_valid(path):
    """내려받은 파일이 HTML 오류 페이지가 아닌지 확인합니다."""
    head = path.read_bytes()[:16]
    if path.suffix == ".pdf":
        return head.startswith(b"%PDF")
    return len(head) > 0 and not head.lstrip().startswith(b"<")


def download(path, urls):
    if path.exists() and looks_valid(path):
        print("이미 준비됨:", path)
        return True
    for url in urls:
        try:
            urllib.request.urlretrieve(url, path)
            if looks_valid(path):
                print("다운로드 완료:", path.name, "←", url[:45], "...")
                return True
        except Exception as error:
            print("실패, 다음 방법 시도:", type(error).__name__)
        path.unlink(missing_ok=True)
    return False


missing = [p for p, urls in SOURCES.items() if not download(p, urls)]

if missing:
    # ③ 마지막 방법: Colab 왼쪽 업로드 창으로 직접 올리기
    print("직접 업로드할 파일:", [p.name for p in missing])
    from google.colab import files
    for name, data in files.upload().items():
        (DATA_DIR / name).write_bytes(data)

assert PDF_PATH.exists() and CSV_PATH.exists(), "데이터 준비 실패"
print("PDF 크기:", PDF_PATH.stat().st_size, "bytes")

In [ ]:
# gradio·pypdf 설치 (1~2분). openai는 Colab에 기본 설치되어 있습니다.
!pip -q install pypdf gradio openai

## 1. 검색기 준비 (워크북 1 코드 재사용)
워크북 1에서 만든 추출 → 청킹 → TF-IDF 검색 코드를 한 번에 실행합니다.

In [ ]:
# 1. PDF 텍스트 추출: 페이지 번호를 메타데이터로 함께 저장합니다.
import re
from pypdf import PdfReader

reader = PdfReader(str(PDF_PATH))
print("전체 페이지 수:", len(reader.pages))

records = []
for page_no, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    text = re.sub(r"\s+", " ", text).strip()  # 줄바꿈·연속 공백 정리
    if text:
        records.append({"source": PDF_PATH.name,
                        "page": page_no,
                        "text": text})

assert records, "텍스트가 없습니다. 스캔 PDF라면 OCR이 필요합니다."
print("텍스트가 있는 페이지 수:", len(records))

In [ ]:
def chunk_words(text, size=60, overlap=15):
    """공백 기준 단어(어절)로 나누고, 이웃 청크가 overlap개를 공유합니다."""
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("0 <= overlap < size 조건이 필요합니다.")
    words = text.split()
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):  # 마지막 단어까지 담았으면 종료
            break
    return chunks


def build_chunks(records, size=60, overlap=15):
    rows = []
    for record in records:
        parts = chunk_words(record["text"], size, overlap)
        for number, part in enumerate(parts, start=1):
            rows.append({"source": record["source"],
                         "page": record["page"],
                         "chunk": number,
                         "text": part})
    return rows


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def build_index(chunks, analyzer="char_wb", ngram_range=(2, 3)):
    """청크 목록으로 TF-IDF 사전을 만들고 문서 벡터를 계산합니다."""
    vectorizer = TfidfVectorizer(analyzer=analyzer,
                                 ngram_range=ngram_range,
                                 sublinear_tf=True)
    matrix = vectorizer.fit_transform([c["text"] for c in chunks])
    return {"chunks": chunks, "vectorizer": vectorizer, "matrix": matrix}


def retrieve(index, question, top_k=3):
    """질문과 가장 비슷한 청크 top_k개를 점수 순으로 돌려줍니다."""
    query = index["vectorizer"].transform([question])
    scores = cosine_similarity(query, index["matrix"]).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [{**index["chunks"][i], "score": float(scores[i])}
            for i in order]


CHUNK_SIZE = 60
OVERLAP = 15
THRESHOLD = 0.16  # 워크북 1의 점수 분포에서 정한 값

index = build_index(build_chunks(records, CHUNK_SIZE, OVERLAP))
print("청크 수:", len(index["chunks"]))

In [ ]:
# 검색 + 임계값 판정 확인
for question in ["RAG는 무엇의 약어인가요?",
                 "API 키는 어디에 저장해야 하나요?",
                 "이 강의의 수업료는 얼마인가요?"]:
    best = retrieve(index, question, top_k=1)[0]
    found = best["score"] >= THRESHOLD
    print(f"{question}\n  → p.{best['page']} 점수 {best['score']:.3f}",
          "근거 있음" if found else "문서에서 확인할 수 없음")

## 2. 프롬프트 구성
검색된 청크에 `[근거 1] (p.2)`처럼 번호와 페이지를 붙여 LLM이 출처를 표시할 수 있게 합니다.

In [ ]:
INSTRUCTIONS = (
    "너는 수업 자료 질의응답 도우미다. "
    "반드시 제공된 근거 안의 내용만 사용해 한국어로 3문장 이내로 답하라. "
    "답의 끝에 사용한 근거의 페이지를 (p.번호) 형식으로 적어라. "
    "근거에 답이 없으면 '문서에서 확인할 수 없습니다.'라고만 답하라."
)


def build_prompt(question, hits):
    blocks = [f"[근거 {n}] ({hit['source']} p.{hit['page']})\n{hit['text']}"
              for n, hit in enumerate(hits, start=1)]
    return "질문: " + question + "\n\n" + "\n\n".join(blocks)


example_hits = retrieve(index, "RAG는 무엇의 약어인가요?", top_k=2)
print(build_prompt("RAG는 무엇의 약어인가요?", example_hits))

## 3. (선택) 수업용 OpenAI API 키 연결
- 키는 **수업 시간에 따로 안내**합니다. 노트북·GitHub·노션에 키를 적지 마세요.
- 방법 A: Colab 왼쪽 🔑(Secrets)에 이름 `OPENAI_API_KEY`로 저장 → 노트북 접근 허용
- 방법 B: 아래 셀 실행 후 나타나는 입력창에 붙여넣기 (화면·파일에 남지 않음)
- 키가 없으면 입력창에서 그냥 **Enter** → 검색 모드로 진행합니다.

In [ ]:
import os
from getpass import getpass

OPENAI_MODEL = "gpt-4o-mini"  # 수업에서 안내한 모델로 바꿀 수 있습니다.

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY") or ""
    except Exception:  # Secrets 미등록·접근 미허용·Colab 외 환경
        api_key = ""
if not api_key:
    try:
        api_key = getpass("수업용 API 키 (없으면 Enter): ").strip()
    except Exception:  # 입력창을 지원하지 않는 실행 환경
        api_key = ""

client = None
if api_key:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    print("LLM 모드 사용 가능 · 모델:", OPENAI_MODEL)
else:
    print("API 키 없음 → 검색 모드로 진행합니다.")

In [ ]:
def llm_answer(question, hits):
    """근거와 질문을 LLM에 보내 답변 문자열을 받습니다."""
    response = client.responses.create(
        model=OPENAI_MODEL,
        instructions=INSTRUCTIONS,
        input=build_prompt(question, hits),
        temperature=0.2,
    )
    return response.output_text.strip()

## 4. 질의응답 함수 `answer()`
1. 질문이 비었는지 확인 → 2. 검색 → 3. 임계값 판정 → 4. (선택) LLM 답변 → 5. 근거 표시

In [ ]:
NOT_FOUND = "문서에서 확인할 수 없습니다."


def format_evidence(hits):
    lines = []
    for n, hit in enumerate(hits, start=1):
        lines.append(f"[근거 {n}] p.{hit['page']} · 점수 {hit['score']:.3f}")
        lines.append(hit["text"])
        lines.append("")
    return "\n".join(lines).strip()


def answer(question, use_llm=False, top_k=3):
    question = (question or "").strip()
    if not question:
        return "질문을 입력하세요.", ""

    hits = retrieve(index, question, top_k=int(top_k))
    evidence = format_evidence(hits)
    best = hits[0]

    if best["score"] < THRESHOLD:
        return (f"**{NOT_FOUND}**\n\n최고 점수 {best['score']:.3f}"
                f" < 임계값 {THRESHOLD}", evidence)

    if use_llm and client is not None:
        try:
            return "**LLM 답변**\n\n" + llm_answer(question, hits), evidence
        except Exception as error:
            note = f"LLM 호출 오류({type(error).__name__}) → 검색 결과만 표시"
            return note + "\n\n" + best["text"], evidence

    mode = "검색 모드 (키 없음)" if use_llm else "검색 모드"
    return (f"**{mode}** · 가장 관련 있는 근거 p.{best['page']}"
            f"\n\n{best['text']}", evidence)


for q in ["RAG는 무엇의 약어인가요?", "오늘 서울 날씨는 어떤가요?"]:
    reply, _ = answer(q)
    print(q, "\n", reply[:120], "\n")

## 5. Gradio 웹 화면
- 셀을 실행하면 아래에 화면이 나타납니다. 예시 질문을 눌러 보세요.
- 화면을 끄려면 셀 실행을 중지(■)하거나 `app.close()`를 실행합니다.

In [ ]:
import gradio as gr

app = gr.Interface(
    fn=answer,
    inputs=[
        gr.Textbox(label="문서 질문", lines=2,
                   placeholder="예: 청크 겹침은 보통 얼마로 설정하나요?"),
        gr.Checkbox(label="LLM 답변 사용 (수업용 키 필요)",
                    value=client is not None),
        gr.Slider(1, 5, value=3, step=1, label="근거 개수 top_k"),
    ],
    outputs=[
        gr.Markdown(label="답변"),
        gr.Textbox(label="검색 근거 (페이지 · 점수)", lines=12),
    ],
    examples=[
        ["RAG는 무엇의 약어인가요?", False, 3],
        ["API 키는 어디에 저장해야 하나요?", False, 3],
        ["이 강의의 수업료는 얼마인가요?", False, 3],
    ],
    title="7주차 PDF RAG 질의응답",
    description=f"문서: {PDF_PATH.name} · 임계값 {THRESHOLD}",
    flagging_mode="never",
)

app.launch()

## 과제 (제출: Gradio 실행 화면 캡처 + 아래 기록표)

| 질문 | 1순위 근거 페이지·점수 | 검색 모드 응답 | LLM 모드 응답 (선택) |
| --- | --- | --- | --- |
| 문서에 있는 질문 ① | | | |
| 문서에 있는 질문 ② | | | |
| 문서에 없는 질문 ③ | | | |

1. 문서에 없는 질문에 시스템이 어떻게 답했는지, 그 이유(임계값·지시문)를 설명하세요.
2. `THRESHOLD`를 0.10과 0.25로 바꾸면 어떤 질문의 판정이 달라지는지 확인하세요.
3. (선택) LLM 답변의 페이지 표시가 검색 근거와 일치하는지 확인하세요.